# Pairs Trading with Spread Residual and Cycle Analysis

Pairs trading usually starts with a spread z-score. De-Time can help separate spread trend drift from residual mean reversion. A spread that trends persistently is often a broken relative-value trade, not a cheap entry.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in [ROOT / "src", ROOT / "examples"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from quant_trading.data import fetch_yahoo_prices, fetch_yahoo_ohlcv, data_audit_report, DEFAULT_UNIVERSES
from quant_trading.features import decompose_one_series, walkforward_decompose, build_feature_table
from quant_trading.signals import (
    trend_pullback_signals,
    residual_mean_reversion_signals,
    turtle_donchian_signals,
    pair_trading_weights,
    cross_sectional_rotation_weights,
    residual_stress_filter,
)
from quant_trading.backtest import backtest_weights, backtest_long_short_signals, summarize_returns

In [ ]:
pair_prices = fetch_yahoo_prices(["KO", "PEP"], start="2016-01-01", cache_dir=ROOT / "examples" / "quant_trading" / "data" / "cache")
weights = pair_trading_weights(pair_prices["KO"], pair_prices["PEP"], lookback=120, entry_z=1.5, exit_z=0.25)
result = backtest_weights(pair_prices, weights, fee_bps=1.0, slippage_bps=2.0)
result.stats_frame()

In [ ]:
spread = np.log(pair_prices["KO"]) - np.log(pair_prices["PEP"])
spread_frame = decompose_one_series(spread.add(100.0), method="STL", period=63, use_log_price=False)
spread_frame[["transformed_price", "trend", "residual", "residual_z"]].tail()

In [ ]:
spread_frame[["transformed_price", "trend"]].plot(figsize=(10, 4), title="KO/PEP spread and De-Time trend")
plt.show()